# Phase 02: Data Cleaning & Preprocessing Pipeline
**Project:** Student Mental Health Risk Prediction  
**Objective:** Transform raw, partially incomplete student records into a leak-free, clean, encoded, and scaled numerical feature space ready for machine learning model training.

### Preprocessing Roadmap Derived from Phase 01 EDA:
1. **Domain Rule Correction:** Resolve logical discrepancies where `Social_Media_Hours > Screen_Time_Hours` (37 records).
2. **Identifier Exclusion:** Drop `Student_ID` to prevent data leakage and spurious primary key memorization.
3. **Stratified Splitting:** Partition into an 80/20 train/test split with stratification on `Mental_Health_Risk` (preserving the 1:1:1 class balance).
4. **Leak-Free Processing:** Fit all imputation and scaling parameters strictly on the training fold (`X_train`) and apply them onto the test fold (`X_test`).
5. **Modular Sub-Pipelines:**
   - **Numerical:** Median imputation + Z-score standardization (`StandardScaler`).
   - **Nominal:** Mode imputation + One-Hot Encoding (`OneHotEncoder`).
   - **Ordinal:** Mode imputation + Explicit Rank Encoding (`OrdinalEncoder`).

## 1. Environment Setup & Dependency Imports

### What this part does:
- Imports data manipulation (`pandas`, `pathlib.Path`) and Scikit-Learn preprocessing modules:
  - `train_test_split`: For stratified data partitioning.
  - `Pipeline` & `ColumnTransformer`: For modular, reproducible feature transformation workflows.
  - `SimpleImputer`: For missing value handling across numerical and categorical columns.
  - `StandardScaler`, `OneHotEncoder`, `OrdinalEncoder`: For feature scaling and categorical encoding.

In [1]:
import sys
sys.path.append("..")

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from src.preprocess import (
    correct_feature,
    target_isolation,
    split_data,
    feats_categorization,
    numericals_pipeline,
    nominals_pipeline,
    ordinals_pipeline,
    preprocessor_col_transformer
)

## 2. Directory Path Configuration & Raw Data Loading

### What this part does:
- Configures dynamic relative filesystem paths pointing to the project parent directory.
- Locates `data/student_mental_health.csv` and ingests the raw dataset ($2,000$ rows, $17$ columns) into `df_clean`.

In [3]:
root = Path.cwd().parent

In [4]:
DATA_PATH = root / "data" / "origine" / "student_mental_health.csv"

In [5]:
df_clean = pd.read_csv(DATA_PATH)

## 3. Cross-Feature Consistency Correction (Domain Rule Enforcement)

### What this part does:
- Resolves the logical inconsistency identified during Phase 01 EDA: **37 records (1.85%)** where reported daily social media hours exceeded total daily screen time.
- Implements a domain reconciliation rule: caps `Screen_Time_Hours` to equal `Social_Media_Hours` for all violating records, ensuring physical subset consistency.

In [6]:
df_clean = correct_feature(df_clean)

## 4. Target Isolation & Identifier Exclusion (Leakage Prevention)

### What this part does:
- Defines the target column: `TARGET = "Mental_Health_Risk"`.
- Excludes `Student_ID` from the feature matrix $\mathbf{X}$ to eliminate target leakage and avoid memorizing arbitrary index tokens.
- Separates explanatory features $\mathbf{X}$ (15 features) from the ground truth target vector $\mathbf{y}$.

In [7]:
X, y = target_isolation(df_clean)

## 5. Stratified Train-Test Splitting (80/20 Partition)

### What this part does:
- Divides the dataset into **80% training set** ($1,600$ observations) and **20% test set** ($400$ observations).
- Uses `random_state=42` for exact reproducibility.
- Enforces `stratify=y` to guarantee that train and test partitions maintain the identical 33/33/34% target class distribution.
- **Data Leakage Safeguard:** Splitting before any imputation or scaling ensures test data remains completely unseen during pipeline fitting.

In [8]:
X_train, X_test, y_train, y_test = split_data(X, y)

## 6. Verification of Train and Test Set Dimensions

### What this part does:
- Confirms the dimensional shape of the generated partitions:
  - $\mathbf{X}_{\text{train}}$: $(1600, 15)$, $\mathbf{y}_{\text{train}}$: $(1600,)$
  - $\mathbf{X}_{\text{test}}$: $(400, 15)$, $\mathbf{y}_{\text{test}}$: $(400,)$

In [9]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1600, 15)
X_test: (400, 15)
y_train: (1600,)
y_test: (400,)


## 7. Class Distribution & Stratification Verification

### What this part does:
- Computes absolute counts and relative percentages for each risk class across both training and test folds.
- Verifies that both splits mirror the population proportions: High (~34.0%), Medium (~33.0%), Low (~33.0%).

In [10]:
train_distribution = pd.DataFrame({
    "Count": y_train.value_counts(),
    "Percentage": y_train.value_counts(normalize=True) * 100
})

test_distribution = pd.DataFrame({
    "Count": y_test.value_counts(),
    "Percentage": y_test.value_counts(normalize=True) * 100
})

print("Training distribution:")
display(train_distribution)

print("Testing distribution:")
display(test_distribution)

Training distribution:


,Count,Percentage
Mental_Health_Risk,,
High,544,34.0
Medium,528,33.0
Low,528,33.0


Testing distribution:


,Count,Percentage
Mental_Health_Risk,,
High,136,34.0
Medium,132,33.0
Low,132,33.0


## 8. Feature Categorization by Data Type & Scale

### What this part does:
- Explicitly isolates the 15 features into three distinct processing groups based on EDA data typing:
  1. **10 Numerical Features:** Continuous/discrete metrics (`Age`, `Year_of_Study`, `Sleep_Hours`, `Study_Hours_Per_Week`, `GPA`, `Social_Media_Hours`, `Screen_Time_Hours`, `Financial_Stress`, `Family_Support`, `Attendance_Rate`).
  2. **3 Nominal Features:** Categorical variables without natural hierarchy (`Gender`, `Part_Time_Job`, `Previous_Mental_Health_History`).
  3. **2 Ordinal Features:** Categorical variables with intrinsic hierarchy (`Physical_Activity`, `Diet_Quality`).

In [11]:
numerical_features, nominal_features, ordinal_features = feats_categorization()

## 9. Numerical Feature Pipeline (Median Imputation & Standardization)

### What this part does:
- Constructs `numeric_pipeline`:
  1. `SimpleImputer(strategy="median")`: Imputes missing continuous entries using the training median (robust against extreme values).
  2. `StandardScaler()`: Centers features to zero mean ($\mu = 0$) and scales to unit variance ($\sigma = 1$).

In [12]:
numeric_pipeline = numericals_pipeline()

## 10. Nominal Feature Pipeline (Mode Imputation & One-Hot Encoding)

### What this part does:
- Constructs `nominal_pipeline`:
  1. `SimpleImputer(strategy="most_frequent")`: Imputes missing nominal entries using the training mode.
  2. `OneHotEncoder(handle_unknown="ignore")`: Expands nominal attributes into binary indicator columns while safely ignoring unseen categories at test time.

In [13]:
nominal_pipeline = nominals_pipeline()

## 11. Ordinal Feature Pipeline (Mode Imputation & Explicit Ordinal Encoding)

### What this part does:
- Constructs `ordinal_pipeline`:
  1. `SimpleImputer(strategy="most_frequent")`: Imputes missing values using the most frequent category.
  2. `OrdinalEncoder(categories=...)`: Maps hierarchical levels to ordered integer ranks:
     - `Physical_Activity`: `Low` (0) < `Moderate` (1) < `High` (2)
     - `Diet_Quality`: `Poor` (0) < `Average` (1) < `Good` (2)

In [14]:
ordinal_pipeline = ordinals_pipeline()


## 12. ColumnTransformer Integration (Unified Preprocessor)

### What this part does:
- Integrates the three dedicated sub-pipelines into a single composite `ColumnTransformer` (`preprocessor`).
- Routes each column subset to its respective transformation pipeline in parallel.

In [ ]:
preprocessor = preprocessor_col_transformer(
    numeric_pipeline,
    numerical_features,
    nominal_pipeline,
    nominal_features,
    ordinal_pipeline,
    ordinal_features,
)

## 13. Preprocessing Pipeline Definition

The preprocessing architecture has been fully defined using
a ColumnTransformer integrating numerical, nominal, and
ordinal transformation pipelines.

The preprocessor is intentionally not fitted at this stage.

It will be integrated with each classifier inside a unified
Scikit-Learn Pipeline during Phase 03.

This ensures that preprocessing parameters are independently
learned within each Cross-Validation training fold, preventing
data leakage.

## 14. Preprocessing Summary & Readiness for Phase 03 (Modeling)

### Accomplishments in Phase 02:

- **Data Ingestion & Cleaning:** Loaded the raw dataset and applied the identified domain consistency correction, subject to the verified definition of `Screen_Time_Hours`.
- **Identifier Exclusion:** Removed `Student_ID` from the feature matrix to avoid using an arbitrary identifier as a predictive feature.
- **Stratified Data Splitting:** Partitioned the dataset into 80% training (1,600 observations) and 20% testing (400 observations), preserving the target class distribution.
- **Feature Categorization:** Organized the 15 input features into three groups: 10 numerical, 3 nominal, and 2 ordinal features.
- **Modular Preprocessing Architecture:** Defined dedicated pipelines for numerical imputation and scaling, nominal imputation and One-Hot Encoding, and ordinal imputation and rank encoding.
- **Leakage Prevention Strategy:** Designed the preprocessing architecture to learn imputation, encoding, and scaling parameters exclusively from training data.

### Preprocessing Execution Strategy:

The preprocessing architecture has been defined but intentionally not fitted at this stage.

The `preprocessor` will be integrated with each classification algorithm inside a unified Scikit-Learn `Pipeline` during Phase 03.

This approach ensures that preprocessing operations are independently fitted within each Cross-Validation training fold, preventing information leakage into validation folds.

### Ready for Phase 03 (Model Training & Evaluation):

The training and testing datasets (`X_train`, `X_test`, `y_train`, and `y_test`) are prepared.

The preprocessing architecture is ready to be integrated with baseline classification algorithms:

- Logistic Regression
- Random Forest
- XGBoost
- Support Vector Machine (SVM)

The next phase will focus on model training, baseline comparison, cross-validation, and hyperparameter optimization.